# Munge sumstats

The workflow creates a missing output directory and reuses an existing one. Existing owned `sumstats.*` artifacts are refused before kernel QC starts unless you set `overwrite=True`, even if the current `output_format` would not produce every sibling. With overwrite enabled, successful reruns remove stale sibling formats that were not produced by the current run, so the directory represents one coherent munge-sumstats run. The default output is `sumstats.parquet`; set `output_format="tsv.gz"` for legacy `sumstats.sumstats.gz` or `output_format="both"` to write both. The CLI path and Python API both go through `SumstatsMunger.run()`, which owns the fixed outputs and delegates low-level parsing/filtering to the kernel. `sumstats.log` is an audit file, so `MungeRunSummary.output_paths` and metadata `output_files` list data artifacts only. Set `use_hm3_snps=True` for the packaged HM3 map, or `sumstats_snps_file` when you want a custom headered SNP keep-list; it filters rows only and does not allele-match or reorder output.

Physical raw inputs are plain, gzip-compressed, or bzip2-compressed whitespace-delimited text. `sumstats_format="auto"` is the default and detects common plain text, old DANER, new DANER, and PGC VCF-style headers. `A1` is the allele that the signed statistic is relative to; `A2` is the counterpart allele. `NEFF` is not treated as `N` automatically, so pass `column_hints={"N_col": "NEFF"}` only when that is appropriate for the analysis.

In [ ]:
import os
import sys

# Set this as the absolute path to the src directory in your environment.
SRC_DIR = "directory/to/src"

if not os.path.isdir(SRC_DIR):
    raise FileNotFoundError(f"Set SRC_DIR to an existing path before running the notebook: {SRC_DIR}")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# import ldsc
from ldsc import MungeConfig, SumstatsMunger

In [ ]:
# Set these explicitly for your environment.
TRAIT_NAME = "trait"  # for labeling tables only
SUMSTATS_FILE = "path/to/your/sumstats.txt"
SUMSTATS_SNPS_FILE = None  # or "path/to/headered_keep_list.tsv.gz"
OUTPUT_DIR = f"directory/for/outputs/{TRAIT_NAME}"


In [ ]:
sumstats = SumstatsMunger().run(
    MungeConfig(
        raw_sumstats_file=SUMSTATS_FILE,
        trait_name=TRAIT_NAME,
        use_hm3_snps=True,
        output_dir=OUTPUT_DIR,
        # sumstats_format="daner-old",  # old DANER: N from FRQ_A_<Ncas> and FRQ_U_<Ncon> headers
        # sumstats_format="daner-new",  # new DANER: per-SNP N from Nca and Nco columns
        # column_hints={"N_col": "NEFF"},  # only if NEFF should be used as N
        # overwrite=True,  # also removes stale unselected sumstats sibling formats
    ),
)


See a small summary:

In [ ]:
sumstats.summary()